# 02 — Data Standardization: dividendos Trading 212

Objetivo: padronizar os tipos e formatos das colunas de `output/dividends.csv` (já enriquecido com o setor) para que os dados fiquem consistentes antes da limpeza e da análise exploratória.

**Entradas**
- `output/dividends.csv` — histórico de dividendos extraído da API da Trading 212
- `output/sectors.csv` — mapa `isin → setor` gerado no notebook `01_expo`

**Pontos levantados no data understanding**
- `paidOn` está como texto (`object`) e tem fusos horários diferentes (`+02:00` e `+03:00`)
- `amount` e `amountInEuro` têm valores idênticos em todas as linhas
- `type` tem um único valor (`DIVIDEND`) e `reference` é um ID único por pagamento

## 1. Imports

In [121]:
import numpy as np
import pandas as pd
from pathlib import Path

In [122]:
import yfinance as yf

## 2. Carregamento dos dados

In [123]:
output_path = Path.cwd().parent / 'output'

df = pd.read_csv(output_path / 'dividends.csv', sep=',')

In [124]:
mapa = pd.read_csv(output_path / 'sectors.csv').set_index('isin')['sector'].to_dict()
df['sector'] = df['isin'].map(mapa)

In [125]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ticker               145 non-null    object 
 1   name                 145 non-null    object 
 2   isin                 145 non-null    object 
 3   currency             145 non-null    object 
 4   amountInEuro         145 non-null    float64
 5   paidOn               145 non-null    object 
 6   type                 145 non-null    object 
 7   grossAmountPerShare  145 non-null    float64
 8   reference            145 non-null    object 
 9   quantity             145 non-null    float64
 10  amount               145 non-null    float64
 11  sector               145 non-null    object 
dtypes: float64(4), object(8)
memory usage: 13.7+ KB


## 3. Remoção de colunas redundantes
`amount` é igual a `amountInEuro` em todas as linhas e `type` tem um único valor (`DIVIDEND`).

In [126]:
# Por tem uma carteira de dividemndos com ativos em diferentes moedas, amount que é o montante líquido recebeido e o amountInEuro também é o mesmo montante mas em euro 
# Vou remover a coluna de amount e manter apenas a coluna amountInEuro
del df['amount']

In [127]:
# Como os dados aqui a serem estudados são dados de dividendos e a coluna type apenas está a indicar isso então vou remover essa coluna
df = df.drop(columns=['type'], errors='ignore')

In [128]:
df.columns

Index(['ticker', 'name', 'isin', 'currency', 'amountInEuro', 'paidOn',
       'grossAmountPerShare', 'reference', 'quantity', 'sector'],
      dtype='object')

## 4. Conversão de `paidOn` para datetime

`paidOn` é a data de pagamento de cada dividendo e a base de qualquer previsão no tempo. Hoje está guardada como texto, com fusos horários diferentes entre linhas.

In [129]:
df['paidOn'] = pd.to_datetime(df['paidOn'], utc=True)

## 5. Montante bruto na moeda original
`quantity × grossAmountPerShare` dá o valor bruto na moeda do ativo (USD ou EUR).

In [130]:
# Criação da coluna de montante bruto do dividndo
df['totalGrossAmountOriginalCurrency'] = df['quantity'] * df['grossAmountPerShare']

In [131]:
df.head()

,ticker,name,isin,currency,amountInEuro,paidOn,grossAmountPerShare,reference,quantity,sector,totalGrossAmountOriginalCurrency
0,SDIVm_EQ,Global X Superdividend (Dist),IE00077FRP95,EUR,0.58,2026-10-30 13:45:23+00:00,0.071554,4890ad14-254e-4878-b330-ed121f5ce758,8.126585,ETF,0.581488
1,JEPIl_EQ,JPMorgan US Equity Premium Income Active (Dist),IE000U5MJOZ6,USD,0.04,2026-10-23 11:27:20+00:00,0.141100,8b78e8b4-8180-4c93-94cc-2c795b5de8bc,0.320549,ETF,0.045229
2,JEPGl_EQ,JPMorgan Global Equity Premium Income Active (...,IE0003UVYC20,USD,0.41,2026-10-23 11:26:23+00:00,0.119300,03721238-a3b8-4d8a-8571-e9d9de4a34c8,3.829508,ETF,0.456860
3,STAG_US_EQ,STAG Industrial,US85254J1025,USD,0.19,2026-10-15 15:12:58+00:00,0.387500,cc81d873-837e-4095-9bad-365af078ef65,0.631032,Real Estate,0.244525
4,EPR_US_EQ,EPR Properties,US26884U1097,USD,1.12,2026-10-15 14:51:00+00:00,0.310000,631c89c0-0bca-4a9c-9ee1-83fe23a5e1b7,4.767652,Real Estate,1.477972


## 6. Taxas de câmbio EUR/USD (yfinance)
`fxEurUsd` = dólares por 1 euro, um valor por dia útil.

In [132]:
min_date = df['paidOn'].min()
max_date = df['paidOn'].max()

In [133]:
fx = yf.Ticker('EURUSD=X').history(start=min_date, end=max_date + pd.Timedelta(days=1)).reset_index()

In [134]:
# Keeping only date and close
fx = fx[['Date', 'Close']].copy()
fx.rename(columns={'Close': 'fxEurUsd'}, inplace=True)
fx['Date'] = fx['Date'].dt.tz_localize(None).dt.normalize()

In [135]:
fx.head(2)

,Date,fxEurUsd
0,2024-07-15,1.088554
1,2024-07-16,1.090156


## 7. Merge dos dividendos com as taxas
`merge_asof` pela data do pagamento: cada pagamento fica com a última taxa disponível até esse dia.

In [136]:
# Creating date column on dataframe
df['payDate'] = df['paidOn'].dt.tz_localize(None).dt.normalize()

In [137]:
merged_data = pd.merge_asof(
    df.sort_values('payDate'),
    fx.sort_values('Date'),
    left_on='payDate',
    right_on='Date',
    direction='backward',
)

In [145]:
# Merged sucessfully total rows keeped
merged_data.shape

(145, 15)

In [144]:
merged_data.columns

Index(['ticker', 'name', 'isin', 'currency', 'amountInEuro', 'paidOn',
       'grossAmountPerShare', 'reference', 'quantity', 'sector',
       'totalGrossAmountOriginalCurrency', 'payDate', 'Date', 'fxEurUsd',
       'grossAmountInEuro'],
      dtype='object')

In [140]:
merged_data['fxEurUsd'].isna().sum() # All fees placed

np.int64(0)

## 8. Montante bruto em euros

In [141]:
merged_data['grossAmountInEuro'] = np.where(
    merged_data['currency'] == 'USD',
    merged_data['totalGrossAmountOriginalCurrency'] / merged_data['fxEurUsd'],
    merged_data['totalGrossAmountOriginalCurrency'],
)

In [142]:
merged_data.head()

,ticker,name,isin,currency,amountInEuro,paidOn,grossAmountPerShare,reference,quantity,sector,totalGrossAmountOriginalCurrency,payDate,Date,fxEurUsd,grossAmountInEuro
0,O_US_EQ,Realty Income,US7561091049,USD,0.04,2024-07-15 11:11:44+00:00,0.263,6aa9ea34-7a8d-4cd7-bfce-1b8f8d415d52,0.201493,Real Estate,0.052993,2024-07-15,2024-07-15,1.088554,0.048682
1,O_US_EQ,Realty Income,US7561091049,USD,0.04,2024-08-15 15:24:11+00:00,0.263,dba63e49-5f8f-477a-a332-8fde1e13de04,0.201493,Real Estate,0.052993,2024-08-15,2024-08-15,1.101322,0.048117
2,OXLC_US_EQ,Oxford Lane Capital,US6915438476,USD,0.22,2024-08-30 11:09:47+00:00,0.090,45daa557-5a76-4845-b87c-4d8e502c0747,3.180450,Financial Services,0.286241,2024-08-30,2024-08-30,1.108045,0.258329
3,O_US_EQ,Realty Income,US7561091049,USD,0.21,2024-09-13 11:39:12+00:00,0.263,de70badc-a717-4892-8571-6995250af640,1.038868,Real Estate,0.273222,2024-09-13,2024-09-13,1.107788,0.246638
4,ARR_US_EQ,ARMOUR Residential REIT,US0423157058,USD,0.24,2024-09-27 12:36:44+00:00,0.240,54db02b3-389d-45d2-b770-5a2f6422dfa1,1.338355,Real Estate,0.321205,2024-09-27,2024-09-27,1.117718,0.287376


In [143]:
merged_data['amountInEuro'] / merged_data['grossAmountInEuro']

0      0.821664
1      0.831301
2      0.851627
3      0.851451
4      0.835143
         ...   
140    0.848357
141    0.869876
142    1.007047
143    0.992402
144    0.997441
Length: 145, dtype: float64

In [150]:
merged_data = merged_data.drop(columns=['reference'])

In [151]:
merged_data.head()

,ticker,name,isin,currency,amountInEuro,paidOn,grossAmountPerShare,quantity,sector,totalGrossAmountOriginalCurrency,payDate,Date,fxEurUsd,grossAmountInEuro
0,O_US_EQ,Realty Income,US7561091049,USD,0.04,2024-07-15 11:11:44+00:00,0.263,0.201493,Real Estate,0.052993,2024-07-15,2024-07-15,1.088554,0.048682
1,O_US_EQ,Realty Income,US7561091049,USD,0.04,2024-08-15 15:24:11+00:00,0.263,0.201493,Real Estate,0.052993,2024-08-15,2024-08-15,1.101322,0.048117
2,OXLC_US_EQ,Oxford Lane Capital,US6915438476,USD,0.22,2024-08-30 11:09:47+00:00,0.090,3.180450,Financial Services,0.286241,2024-08-30,2024-08-30,1.108045,0.258329
3,O_US_EQ,Realty Income,US7561091049,USD,0.21,2024-09-13 11:39:12+00:00,0.263,1.038868,Real Estate,0.273222,2024-09-13,2024-09-13,1.107788,0.246638
4,ARR_US_EQ,ARMOUR Residential REIT,US0423157058,USD,0.24,2024-09-27 12:36:44+00:00,0.240,1.338355,Real Estate,0.321205,2024-09-27,2024-09-27,1.117718,0.287376
